# CS224N · Lecture 6 — LSTM RNNs and Neural Machine Translation

**Slides:** [cs224n-spr2024-lecture06-fancy-rnn.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture06-fancy-rnn.pdf)  
**Prerequisite:** the Lecture 5 notebook (RNN-LMs, BPTT, vanishing/exploding gradients).

### What you will build
| § | Topic | You will implement / measure |
|---|---|---|
| 1 | Vanishing and exploding gradients (recap) | Gradient flow through time, measured at the input |
| 2 | **Long Short-Term Memory (LSTM)** | LSTM forward + backward from scratch, gradient-checked; the forget-gate bias |
| 2.5 | Is vanishing gradient just an RNN problem? Residual, dense, highway connections | Gradient flow in a deep net with/without residuals |
| 2.6 | 🔎 The GRU | Equations + comparison |
| 3 | Other uses of RNNs | — |
| 4 | **Bidirectional and multi-layer RNNs** | A BiLSTM tagger that solves what a left-to-right one provably can't |
| 5 | Machine translation: history, SMT | A word-substitution baseline (1950s style) |
| 6 | **Neural machine translation: sequence-to-sequence** | An LSTM encoder–decoder from scratch |
| 7 | PyTorch versions | `nn.LSTM`, bidirectional, multi-layer, seq2seq |

The **practical takeaways** the lecture ends with, which this notebook demonstrates one by one:
1. **LSTMs are powerful.** 2. **Clip your gradients.** 3. **Use bidirectionality when possible.** 4. **Encoder–decoder NMT systems work very well.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

class Adam:
    def __init__(self, params, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            params[k] -= self.lr * (self.m[k] / (1 - self.b1 ** self.t)) / (np.sqrt(self.v[k] / (1 - self.b2 ** self.t)) + self.eps)

def clip_grads(g, max_norm=5.0):
    norm = math.sqrt(sum((v ** 2).sum() for v in g.values()))
    if norm > max_norm:
        for k in g:
            g[k] *= max_norm / norm
    return norm

---
## 1. Recap: language models, perplexity, and the gradient problem

* **Language model:** a system that predicts the next word. **RNN:** takes sequential input of any length, applies the same weights on each step, can optionally produce output on each step. **RNN ≠ LM**: RNNs are used for much more, and LMs can be built with other models (n-grams, Transformers).
* **Perplexity** $= \exp(J(\theta))$, the exponential of the cross-entropy loss. Lower is better. RNNs greatly improved perplexity over n-gram models.

**Vanishing gradients.** $\frac{\partial J^{(i)}}{\partial h^{(j)}}$ is a product of $i-j$ Jacobians $\frac{\partial h^{(t)}}{\partial h^{(t-1)}} = \text{diag}(\sigma'(\cdot))W_h$. In the linear case this is $W_h^{\,i-j}$: if all eigenvalues of $W_h$ are below 1 in magnitude, the gradient shrinks **exponentially** with distance. As a result, model weights are basically updated only with respect to **near effects**, not long-term effects. **In practice, a simple RNN will only condition ~7 tokens back** (a vague rule of thumb).

**Exploding gradients** cause bad updates (*"suddenly you're in Iowa"*), even Inf/NaN. **Gradient clipping** fixes this: if $\|g\|$ exceeds a threshold, rescale $g$ to the threshold. (See the "cliff" demo in the L5 notebook.)

**How to fix vanishing gradients?** The main problem is that it's too difficult for the RNN to learn to preserve information over many timesteps. In a vanilla RNN, the **hidden state is constantly being rewritten**:
$$h^{(t)} = \sigma\big(W_h h^{(t-1)} + W_x x^{(t)} + b\big)$$
**Could we design an RNN with separate memory which is *added to*?**

### 1.1 A testbed that isolates long-range memory
In L5, a vanilla RNN learned long-distance agreement, because the training data also contained short-distance examples that bootstrapped learning. To isolate the gradient problem, we use a task where **every** training example has a long gap, so there's no short-range signal to learn from first (in the spirit of Hochreiter & Schmidhuber's original experiments):

> input: `[symbol] d d d d … d d [query]` → output at the end: `symbol` (one of 4)

Only the first token matters, and the loss is computed only at the last step. Chance is 25%.

In [ ]:
N_SYM, N_DIST = 4, 7                     # tokens 0..3 = symbols to remember, 4..10 = distractors, 11 = query
V_MEM = N_SYM + N_DIST + 1
QUERY = V_MEM - 1

def memory_batch(T, n, r):
    X = r.integers(N_SYM, N_SYM + N_DIST, size=(n, T + 2))
    target = r.integers(0, N_SYM, size=n)
    X[:, 0] = target
    X[:, -1] = QUERY
    return X, target

X_demo, y_demo = memory_batch(10, 3, np.random.default_rng(0))
print(X_demo, "-> targets", y_demo)

---
## 2. Long Short-Term Memory RNNs (LSTMs)

* A type of RNN proposed by **Hochreiter & Schmidhuber (1997)** as a solution to the vanishing gradient problem.
* Everyone cites that paper, but a crucial part of the modern LSTM, the **forget gate**, is from **Gers et al. (2000)** 💜.
* It only started to be recognized as promising through the work of Schmidhuber's student **Alex Graves** c. 2006 (who also invented **CTC**, connectionist temporal classification, for speech recognition).
* It became really well known after Hinton brought it to Google in 2013 (Graves had been a postdoc with Hinton).

### 2.1 The idea
* On step $t$, there is a **hidden state** $h^{(t)}$ and a **cell state** $c^{(t)}$. Both are vectors of length $n$.
* **The cell stores long-term information.** The LSTM can **read, erase, and write** information from the cell; the cell becomes conceptually rather like RAM in a computer.
* The selection of which information is erased/written/read is controlled by three corresponding **gates**. Gates are also vectors of length $n$, computed things whose values are probabilities: on each timestep, each element of a gate can be **open (1), closed (0), or somewhere in between**.
* The gates are **dynamic**: their value is computed from the current context.

### 2.2 The equations
We have a sequence of inputs $x^{(t)}$, and compute a sequence of hidden states $h^{(t)}$ and cell states $c^{(t)}$. On timestep $t$:

| | Equation | Role |
|---|---|---|
| **Forget gate** | $f^{(t)} = \sigma\big(W_f h^{(t-1)} + U_f x^{(t)} + b_f\big)$ | controls what is kept vs. forgotten from the previous cell state |
| **Input gate** | $i^{(t)} = \sigma\big(W_i h^{(t-1)} + U_i x^{(t)} + b_i\big)$ | controls what parts of the new cell content are written to the cell |
| **Output gate** | $o^{(t)} = \sigma\big(W_o h^{(t-1)} + U_o x^{(t)} + b_o\big)$ | controls what parts of the cell are output to the hidden state |
| **New cell content** | $\tilde c^{(t)} = \tanh\big(W_c h^{(t-1)} + U_c x^{(t)} + b_c\big)$ | the new content to be written to the cell |
| **Cell state** | $c^{(t)} = f^{(t)}\odot c^{(t-1)} + i^{(t)}\odot\tilde c^{(t)}$ | erase ("forget") some content from the last cell state, and write ("input") some new content |
| **Hidden state** | $h^{(t)} = o^{(t)}\odot\tanh c^{(t)}$ | read ("output") some content from the cell |

All gate values are between 0 and 1 (sigmoid); gates are applied with the element-wise (Hadamard) product $\odot$. (Chris Olah's *Understanding LSTM Networks* post has the classic diagram.)

**"The + sign is the secret!"** The cell update is **additive**: $c^{(t)} = f\odot c^{(t-1)} + \dots$, not a rewrite through a matrix and a squashing function.

### 2.3 How does the LSTM solve vanishing gradients?
* The LSTM architecture makes it **much easier** for an RNN to preserve information over many timesteps. E.g., if the forget gate is set to 1 for a cell dimension and the input gate set to 0, the information in that cell is **preserved indefinitely**.
* In contrast, it's harder for a vanilla RNN to learn a recurrent weight matrix $W_h$ that preserves info in the hidden state.
* In practice, you get about **100 timesteps rather than about 7**.
* However, there are alternative ways of creating more direct and linear pass-through connections in models for long-distance dependencies (attention, L7–8).

**In gradient terms** (🔎): along the cell path, $\frac{\partial c^{(t)}}{\partial c^{(t-1)}} = \text{diag}(f^{(t)})$ (plus indirect terms through the gates). There is no repeated multiplication by a weight matrix and no squashing derivative, just the forget gate. If $f\approx1$, the gradient flows back nearly unchanged. This is the same "identity shortcut" idea as residual connections (§2.5).

### 2.4 Implementing an LSTM layer from scratch
Following common practice, we stack the four gates' weights into one matrix $W\in\mathbb{R}^{4n\times(d+n)}$ applied to $[x^{(t)}; h^{(t-1)}]$. This is the same math as the table above, with $[U_f\ W_f]$ etc. as row blocks. The layer supports padded batches via a **mask**: at padding positions the state is carried over unchanged. It also accepts gradients for **every** hidden state (needed for tagging, attention, and stacking) plus the final state.

In [ ]:
def lstm_init(d, H, r, forget_bias=1.0):
    W = r.normal(0, 1 / np.sqrt(d + H), (4 * H, d + H))
    b = np.zeros(4 * H)
    b[:H] = forget_bias                      # gate order in our stacked matrix: f, i, o, c~
    return W, b

def lstm_forward(W, b, X, mask, h0, c0):
    # X: (B, T, d) inputs; mask: (B, T) 1 = real token. Returns all hidden states (B, T, H), final h, c, cache.
    B, T, d = X.shape; H = h0.shape[1]
    h, c = h0, c0
    Hs = np.zeros((B, T, H)); cache = []
    for t in range(T):
        xh = np.concatenate([X[:, t], h], axis=1)
        a = xh @ W.T + b
        f, i, o = sigmoid(a[:, :H]), sigmoid(a[:, H:2 * H]), sigmoid(a[:, 2 * H:3 * H])
        g = np.tanh(a[:, 3 * H:])                        # new cell content c~
        c_new = f * c + i * g                            # the additive update ("the + sign is the secret")
        tc = np.tanh(c_new)
        h_new = o * tc
        m = mask[:, t:t + 1]
        cache.append((xh, f, i, o, g, c, tc, m))
        h = m * h_new + (1 - m) * h                      # padding: carry the state through unchanged
        c = m * c_new + (1 - m) * c
        Hs[:, t] = h
    return Hs, h, c, cache

def lstm_backward(W, dHs, dhT, dcT, cache, d):
    # dHs: gradient w.r.t. every hidden state (B, T, H); dhT, dcT: w.r.t. the final state.
    B, T, H = dHs.shape
    dX = np.zeros((B, T, d)); gW = np.zeros_like(W); gb = np.zeros(W.shape[0])
    dh, dc = dhT.copy(), dcT.copy()
    for t in reversed(range(T)):
        xh, f, i, o, g, c_prev, tc, m = cache[t]
        dh = dh + dHs[:, t]
        dh_new, dc_in = m * dh, m * dc                    # gradient into this step's computation
        dh_prev, dc_prev = (1 - m) * dh, (1 - m) * dc     # gradient that skips it (padding)
        do = dh_new * tc
        dct = dh_new * o * (1 - tc ** 2) + dc_in          # total gradient on c_t
        df, di, dg = dct * c_prev, dct * g, dct * i
        dc_prev = dc_prev + dct * f                       # <- the cell path: just multiply by f
        da = np.concatenate([df * f * (1 - f), di * i * (1 - i), do * o * (1 - o), dg * (1 - g ** 2)], axis=1)
        gW += da.T @ xh; gb += da.sum(0)
        dxh = da @ W
        dX[:, t] = dxh[:, :d]
        dh_prev = dh_prev + dxh[:, d:]
        dh, dc = dh_prev, dc_prev
    return dX, dh, dc, gW, gb

# Vanilla (Elman) RNN layer with the same interface, for comparisons.
def rnn_init(d, H, r, scale=1.0):
    return np.concatenate([r.normal(0, 1 / np.sqrt(d), (H, d)), r.normal(0, scale / np.sqrt(H), (H, H))], 1), np.zeros(H)

def rnn_forward(W, b, X, mask, h0, c0=None):
    B, T, d = X.shape; h = h0; Hs = np.zeros((B, T, h0.shape[1])); cache = []
    for t in range(T):
        xh = np.concatenate([X[:, t], h], 1); hn = np.tanh(xh @ W.T + b); m = mask[:, t:t + 1]
        cache.append((xh, hn, m)); h = m * hn + (1 - m) * h; Hs[:, t] = h
    return Hs, h, None, cache

def rnn_backward(W, dHs, dhT, dcT, cache, d):
    B, T, H = dHs.shape; dX = np.zeros((B, T, d)); gW = np.zeros_like(W); gb = np.zeros(W.shape[0]); dh = dhT.copy()
    for t in reversed(range(T)):
        xh, hn, m = cache[t]; dh = dh + dHs[:, t]
        dz = m * dh * (1 - hn ** 2); dh_prev = (1 - m) * dh
        gW += dz.T @ xh; gb += dz.sum(0); dxh = dz @ W
        dX[:, t] = dxh[:, :d]; dh = dh_prev + dxh[:, d:]
    return dX, dh, None, gW, gb

**Gradient check** (always!). We check both layers with a random scalar loss $L = \sum_t \langle R_t, h^{(t)}\rangle$, which sends a gradient into every hidden state, with padding in the batch.

In [ ]:
r = np.random.default_rng(1)
B_, T_, d_, H_ = 3, 5, 4, 3
Xg = r.normal(size=(B_, T_, d_)); Mg = np.ones((B_, T_)); Mg[1, 3:] = 0; Mg[2, 4:] = 0   # padded batch
R = r.normal(size=(B_, T_, H_))
h0 = r.normal(size=(B_, H_)) * 0.1; c0 = r.normal(size=(B_, H_)) * 0.1

for name, init, fwd, bwd in [("LSTM", lstm_init, lstm_forward, lstm_backward), ("RNN", rnn_init, rnn_forward, rnn_backward)]:
    W, b = init(d_, H_, r)
    loss_fn = lambda W, b, X: (fwd(W, b, X, Mg, h0, c0)[0] * R).sum()
    Hs, hT, cT, cache = fwd(W, b, Xg, Mg, h0, c0)
    dX, _, _, gW, gb = bwd(W, R, np.zeros_like(hT), np.zeros_like(hT), cache, d_)
    worst = 0.0
    for P, G in [(W, gW), (b, gb), (Xg, dX)]:
        for idx in list(np.ndindex(P.shape))[::3]:
            old = P[idx]
            P[idx] = old + 1e-6; lp = loss_fn(W, b, Xg)
            P[idx] = old - 1e-6; lm_ = loss_fn(W, b, Xg)
            P[idx] = old
            worst = max(worst, abs((lp - lm_) / 2e-6 - G[idx]))
    print(f"{name}: max |analytic - numerical| over W, b and inputs = {worst:.1e}")

**The cell as memory.** If we hand-set the forget gate to 1 and the input gate to 0 for one cell dimension, whatever is stored there is preserved **indefinitely**, regardless of the inputs. Let's verify with the actual layer by setting biases to saturate the gates:

In [ ]:
r = np.random.default_rng(0)
d_, H_ = 4, 2
W, b = lstm_init(d_, H_, r)
W[:, :] *= 0.0                         # no input/recurrent influence on gates: the biases decide everything
b[:H_] = [20, -20]                     # forget gate: dim 0 open (f=1), dim 1 closed (f=0)
b[H_:2 * H_] = [-20, -20]              # input gate closed (i=0): never write
b[2 * H_:3 * H_] = 20                  # output gate open
c0 = np.array([[0.9, 0.9]])
Xs = r.normal(size=(1, 200, d_))
Hs, hT, cT, _ = lstm_forward(W, b, Xs, np.ones((1, 200)), np.zeros((1, H_)), c0)
print("cell state after 200 steps:", cT[0].round(4), "-> dim 0 (f=1, i=0) kept its value; dim 1 (f=0) erased it")

### 2.5 Gradient flow through time, measured
Let's measure how much gradient from the loss at the **last** step reaches the **first** input, for an untrained vanilla RNN at different recurrent-weight scales and for an untrained LSTM at different **forget-gate biases**. (Ratio = gradient norm at the first input / at the last input, 50 steps apart.)

In [ ]:
def gradient_ratio(kind, T=50, d=16, H=32, seed=0, **kw):
    r = np.random.default_rng(seed)
    E = r.normal(0, 0.1, (V_MEM, d))
    X, _ = memory_batch(T, 256, r)
    Xe = E[X]; M = np.ones(X.shape)
    if kind == "lstm":
        W, b = lstm_init(d, H, r, forget_bias=kw["forget_bias"]); fwd, bwd = lstm_forward, lstm_backward
    else:
        W, b = rnn_init(d, H, r, scale=kw["scale"]); fwd, bwd = rnn_forward, rnn_backward
    Hs, hT, cT, cache = fwd(W, b, Xe, M, np.zeros((256, H)), np.zeros((256, H)))
    dHs = np.zeros_like(Hs)
    dX, *_ = bwd(W, dHs, r.normal(size=hT.shape), np.zeros_like(hT), cache, d)   # random gradient at the last state
    norms = np.linalg.norm(dX, axis=2).mean(0)
    return norms[0] / norms[-1]

for scale in [0.5, 1.0, 1.5]:
    print(f"vanilla RNN, W_h ~ N(0, ({scale}^2)/H):  gradient ratio first/last input = {gradient_ratio('rnn', scale=scale):.2e}")
for fb in [0.0, 1.0, 3.0]:
    print(f"LSTM, forget-gate bias {fb}:               gradient ratio first/last input = {gradient_ratio('lstm', forget_bias=fb):.2e}")

Two important, practical observations:
* **A vanilla RNN's gradient flow is very sensitive to the scale of $W_h$.** Small recurrent weights vanish fast; weights near spectral radius 1 preserve the signal for a while at initialization. (But training changes $W_h$, and the tanh saturates, so this doesn't last.)
* **An LSTM's memory is only as good as its forget gate.** With bias 0, $f\approx\sigma(0)=0.5$, and the cell path multiplies the gradient by ~0.5 per step: $0.5^{50}\approx10^{-15}$. That's *worse* than a vanilla RNN! A **positive forget-gate bias** (Gers et al. 2000; Jozefowicz et al. 2015 recommend 1 or more) makes $f\approx1$ at the start of training, so the cell path is open by default. PyTorch initializes all LSTM biases near zero; for long dependencies, setting the forget bias yourself is a well-known trick.

### 2.6 Training on the long-gap memory task
Now we actually train, with the same data and optimizer for every model, and report accuracy every 250 steps (chance = 25%). We train **four** models: a vanilla RNN with small recurrent weights (which vanished in §2.5), a vanilla RNN with well-scaled weights, and LSTMs with forget-gate bias 1 and 3.

In [ ]:
def train_memory_model(kind, T, steps, seed=0, d=16, H=32, forget_bias=3.0, rnn_scale=1.0, lr=3e-3, batch=64):
    r = np.random.default_rng(seed)
    p = {"E": r.normal(0, 0.1, (V_MEM, d)), "U": r.normal(0, 1 / np.sqrt(H), (N_SYM, H)), "c": np.zeros(N_SYM)}
    if kind == "lstm":
        p["W"], p["b"] = lstm_init(d, H, r, forget_bias); fwd, bwd = lstm_forward, lstm_backward
    else:
        p["W"], p["b"] = rnn_init(d, H, r, scale=rnn_scale); fwd, bwd = rnn_forward, rnn_backward
    opt = Adam(p, lr=lr)
    def run(X, y, grad=True):
        n = len(y); Xe = p["E"][X]; M = np.ones(X.shape)
        Hs, hT, cT, cache = fwd(p["W"], p["b"], Xe, M, np.zeros((n, H)), np.zeros((n, H)))
        P = softmax(hT @ p["U"].T + p["c"])
        loss = -np.log(P[np.arange(n), y] + 1e-12).mean()
        if not grad:
            return (P.argmax(1) == y).mean()
        G = P.copy(); G[np.arange(n), y] -= 1; G /= n
        g = {"U": G.T @ hT, "c": G.sum(0)}
        dX, _, _, g["W"], g["b"] = bwd(p["W"], np.zeros_like(Hs), G @ p["U"], np.zeros_like(hT), cache, d)
        g["E"] = np.zeros_like(p["E"]); np.add.at(g["E"], X, dX)
        return loss, g
    curve = []
    for step in range(1, steps + 1):
        X, y = memory_batch(T, batch, r)
        _, g = run(X, y); clip_grads(g); opt.step(p, g)
        if step % 250 == 0:
            curve.append(run(*memory_batch(T, 500, r), grad=False))
    return curve

T_GAP, STEPS = 60, 1000
t0 = time.time()
curves = {}
for name, kw in [("vanilla RNN, W_h scale 0.5", dict(kind="rnn", rnn_scale=0.5)),
                 ("vanilla RNN, W_h scale 1.0", dict(kind="rnn", rnn_scale=1.0)),
                 ("LSTM, forget bias 1", dict(kind="lstm", forget_bias=1.0)),
                 ("LSTM, forget bias 3", dict(kind="lstm", forget_bias=3.0))]:
    curves[name] = train_memory_model(T=T_GAP, steps=STEPS, **kw)
    print(f"{name:22s} accuracy every 250 steps: {np.round(curves[name], 2)}   ({time.time() - t0:.0f}s so far)")

In [ ]:
plt.figure(figsize=(7, 4))
for name, c in curves.items():
    plt.plot(np.arange(1, len(c) + 1) * 250, c, "o-", label=name)
plt.axhline(1 / N_SYM, color="grey", ls="--", label="chance")
plt.xlabel("training step"); plt.ylabel("accuracy"); plt.ylim(0, 1.05)
plt.title(f"Remember the first symbol across a gap of {T_GAP} steps"); plt.legend(); plt.show()

**What this shows, and an honest footnote.** Training follows the gradient-flow measurements from §2.5. The models whose gradient vanished at initialization (the small-weight RNN, and the LSTM with a weak forget bias) don't learn the 60-step dependency: they plateau between chance (25%) and 50%, i.e., at best they partially separate the symbols. The ones whose gradient survives reach 100%.

You might have expected the well-initialized vanilla RNN to fail too. On this simple task, with Adam, gradient clipping, and a recurrent matrix with spectral radius ≈ 1, it usually succeeds. (In our runs it solved a gap of 60 on 4 of 5 random seeds; it also handled a harder version where the distractors include look-alike symbols.) The LSTM's practical advantage is clearest on **real** data with many interacting long- and short-range dependencies. On real language modeling, LSTMs beat vanilla RNNs by large perplexity margins (Jozefowicz et al. 2015), and the gating makes them far less sensitive to initialization and learning rate than a vanilla RNN, whose success here depended on landing near the "edge of chaos."

Two lessons worth more than the headline:
* **The LSTM's advantage depends on initialization.** A badly initialized LSTM (forget bias ≈ 0) vanishes faster than a vanilla RNN.
* **Always give the baseline a fair chance.** A well-tuned simple model often closes much of the gap to a fancy one. Many published "improvements" shrink once the baseline is tuned as carefully (Melis, Dyer & Blunsom 2018 found exactly this for LSTM language models).

### 2.7 Is vanishing/exploding gradient just an RNN problem?
**No!** It can be a problem for all neural architectures (including feed-forward and convolutional networks), especially very deep ones. Due to the chain rule and the choice of nonlinearity, the gradient can become vanishingly small as it backpropagates, so lower layers are learned very slowly (hard to train).

**Another solution: add more direct connections** (letting the gradient flow):
* **Residual connections**, aka "ResNet" (He et al. 2015), also known as skip connections: $F(x) + x$. The **identity connection preserves information by default**, which makes deep networks much easier to train.
* **Dense connections**, aka "DenseNet" (Huang et al. 2017): directly connect each layer to all future layers!
* **Highway connections**, aka "HighwayNet" (Srivastava et al. 2015): like residual connections, but the identity connection vs. the transformation layer is controlled by a **dynamic gate**: $y = g\odot F(x) + (1-g)\odot x$. Inspired by LSTMs, but applied to deep feed-forward/convolutional networks.

**Conclusion:** though vanishing/exploding gradients are a general problem, **RNNs are particularly unstable due to the repeated multiplication by the same weight matrix** (Bengio et al. 1994).

Let's measure the residual effect on a 50-layer tanh network:

In [ ]:
def deep_grad_norms(depth=50, width=64, residual=False, seed=0):
    r = np.random.default_rng(seed)
    Ws = [r.normal(0, 0.5 / np.sqrt(width), (width, width)) for _ in range(depth)]   # deliberately small weights
    h = r.normal(size=(32, width)); cache = []
    for W in Ws:
        z = h @ W.T; a = np.tanh(z); cache.append(a)
        h = h + a if residual else a
    G = r.normal(size=h.shape); norms = []
    for W, a in zip(reversed(Ws), reversed(cache)):
        Ga = G * (1 - a ** 2) @ W
        G = G + Ga if residual else Ga          # the identity path passes G straight through
        norms.append(np.linalg.norm(G))
    return norms[::-1]

plt.figure(figsize=(7, 4))
for res in [False, True]:
    plt.semilogy(range(1, 51), deep_grad_norms(residual=res), label="with residual connections" if res else "plain")
plt.xlabel("layer (1 = input side)"); plt.ylabel("gradient norm"); plt.legend()
plt.title("50-layer tanh network: gradient reaching each layer"); plt.show()

### 2.8 🔎 The GRU (Cho et al. 2014)
Lecture 5 promised "other RNN flavors like LSTM and GRU". The **Gated Recurrent Unit** is a simpler alternative with no separate cell state and two gates:
$$u^{(t)} = \sigma(W_u h^{(t-1)} + U_u x^{(t)} + b_u)\quad\text{(update gate)}\qquad r^{(t)} = \sigma(W_r h^{(t-1)} + U_r x^{(t)} + b_r)\quad\text{(reset gate)}$$
$$\tilde h^{(t)} = \tanh\big(W_h(r^{(t)}\odot h^{(t-1)}) + U_h x^{(t)} + b_h\big),\qquad h^{(t)} = (1-u^{(t)})\odot h^{(t-1)} + u^{(t)}\odot\tilde h^{(t)}$$
The update gate does the job of the LSTM's forget *and* input gates together (they're tied: keep $1-u$, write $u$), and the hidden state itself is the additively updated memory. **LSTM vs. GRU:** the GRU is quicker to compute and has fewer parameters (3 weight blocks instead of 4). There's no conclusive evidence that one consistently performs better than the other (Chung et al. 2014; Jozefowicz et al. 2015). The LSTM is a good default, especially with lots of data and long dependencies.

---
## 3. Other uses of RNNs

* **Sequence tagging** (POS tagging, NER): an output at every step.
* **Sentence encoder** (e.g., sentiment classification): compute a **sentence encoding**. Basic way: use the **final hidden state**. Usually better: take the **element-wise max or mean of all hidden states**.
* **Conditional language models** that generate text based on other information: speech recognition (audio → text), **machine translation**, summarization. → §6

---
## 4. Bidirectional and multi-layer RNNs

### 4.1 Motivation
Task: sentiment classification of *"the movie was **terribly** exciting !"* We can regard the hidden state at *terribly* as a representation of the word "terribly" **in the context of this sentence**: a **contextual representation**. But these contextual representations only contain information about the **left context** (*"the movie was"*). What about the right context? In this example, *exciting* is in the right context, and it **modifies the meaning of "terribly"** (from negative to positive).

### 4.2 Bidirectional RNNs
On timestep $t$:
$$\text{Forward RNN:}\quad \overrightarrow{h}^{(t)} = \text{RNN}_{\text{FW}}\big(\overrightarrow{h}^{(t-1)}, x^{(t)}\big)$$
$$\text{Backward RNN:}\quad \overleftarrow{h}^{(t)} = \text{RNN}_{\text{BW}}\big(\overleftarrow{h}^{(t+1)}, x^{(t)}\big)$$
$$\text{Concatenated hidden states:}\quad h^{(t)} = \big[\overrightarrow{h}^{(t)};\overleftarrow{h}^{(t)}\big]$$
"RNN" here is a general notation meaning "compute one step of the RNN": it could be a simple RNN or an LSTM step. **Generally, the two RNNs have separate weights.** We regard $h^{(t)}$ as "the hidden state" of the bidirectional RNN; it's what we pass on to the next parts of the network.

**Note:** bidirectional RNNs are only applicable if you have access to the **entire input sequence**. They are **not applicable to language modeling**, because in LM you only have left context available. If you do have the entire input sequence (e.g., any kind of encoding), **bidirectionality is powerful: you should use it by default.** For example, **BERT** (Bidirectional Encoder Representations from Transformers) is built on bidirectionality (L9).

### 4.3 Experiment: a task that left-to-right models provably can't solve
We tag each word of short reviews with its **contextual polarity**. An intensifier like *terribly / awfully / incredibly / really* takes the polarity of the **adjective that follows it**. A left-to-right tagger has to commit to a tag for *terribly* before it has seen *exciting* or *boring*, so it can't beat 50% on intensifiers. A bidirectional one can.

In [ ]:
SUBJ = [("the", "movie"), ("the", "plot"), ("this", "film"), ("the", "ending"), ("that", "show")]
COP = ["was", "is", "seemed"]
ADV = ["terribly", "awfully", "incredibly", "really"]
POS_ADJ = ["exciting", "good", "funny", "moving"]
NEG_ADJ = ["boring", "bad", "dull", "slow"]
TAGS = ["O", "POS", "NEG"]

def review():
    det, noun = random.choice(SUBJ)
    positive = random.random() < 0.5
    adj = random.choice(POS_ADJ if positive else NEG_ADJ)
    words = [det, noun, random.choice(COP), random.choice(ADV), adj, "!"]
    pol = "POS" if positive else "NEG"
    return words, ["O", "O", "O", pol, pol, "O"]       # the intensifier inherits the adjective's polarity

random.seed(0)
reviews = [review() for _ in range(3000)]
tag_vocab = sorted({w for s, _ in reviews for w in s})
tw2i = {w: i for i, w in enumerate(tag_vocab)}
Xtag = np.array([[tw2i[w] for w in s] for s, _ in reviews])
Ytag = np.array([[TAGS.index(t) for t in tags] for _, tags in reviews])
for s, t in reviews[:3]:
    print(" ".join(f"{w}/{tg}" for w, tg in zip(s, t)))

In [ ]:
def init_tagger(bidirectional, d=16, H=24, seed=0):
    r = np.random.default_rng(seed)
    p = {"E": r.normal(0, 0.1, (len(tag_vocab), d))}
    p["Wf"], p["bf"] = lstm_init(d, H, r)
    if bidirectional:
        p["Wb"], p["bb"] = lstm_init(d, H, r)
    out_dim = 2 * H if bidirectional else H
    p["U"], p["c"] = r.normal(0, 1 / np.sqrt(out_dim), (len(TAGS), out_dim)), np.zeros(len(TAGS))
    return p

def tagger_fb(p, X, Y, need_grad=True):
    B, T = X.shape; d = p["E"].shape[1]; H = p["bf"].shape[0] // 4
    Xe = p["E"][X]; M = np.ones((B, T)); zeros = np.zeros((B, H))
    Hf, _, _, cf = lstm_forward(p["Wf"], p["bf"], Xe, M, zeros, zeros)
    bi = "Wb" in p
    if bi:
        Hb_rev, _, _, cb = lstm_forward(p["Wb"], p["bb"], Xe[:, ::-1], M, zeros, zeros)   # run right-to-left
        Hcat = np.concatenate([Hf, Hb_rev[:, ::-1]], axis=2)                              # align and concatenate
    else:
        Hcat = Hf
    P = softmax(Hcat @ p["U"].T + p["c"])
    Bi, Ti = np.meshgrid(np.arange(B), np.arange(T), indexing="ij")
    loss = -np.log(P[Bi, Ti, Y] + 1e-12).mean()
    if not need_grad:
        return loss, P
    G = P.copy(); G[Bi, Ti, Y] -= 1; G /= B * T
    g = {"U": np.einsum("btk,bth->kh", G, Hcat), "c": G.sum((0, 1))}
    dH = G @ p["U"]
    dXf, _, _, g["Wf"], g["bf"] = lstm_backward(p["Wf"], dH[:, :, :H], zeros, zeros, cf, d)
    dX = dXf
    if bi:
        dXb_rev, _, _, g["Wb"], g["bb"] = lstm_backward(p["Wb"], dH[:, ::-1, H:], zeros, zeros, cb, d)
        dX = dX + dXb_rev[:, ::-1]
    g["E"] = np.zeros_like(p["E"]); np.add.at(g["E"], X, dX)
    return loss, g

n_tr = 2500
for bidirectional in [False, True]:
    p = init_tagger(bidirectional); opt = Adam(p, lr=5e-3); r = np.random.default_rng(0)
    for epoch in range(8):
        perm = r.permutation(n_tr)
        for k in range(0, n_tr, 50):
            idx = perm[k:k + 50]
            _, g = tagger_fb(p, Xtag[idx], Ytag[idx]); clip_grads(g); opt.step(p, g)
    _, P = tagger_fb(p, Xtag[n_tr:], Ytag[n_tr:], False)
    pred = P.argmax(-1)
    acc_adv = (pred[:, 3] == Ytag[n_tr:, 3]).mean(); acc_adj = (pred[:, 4] == Ytag[n_tr:, 4]).mean()
    print(f"{'bidirectional' if bidirectional else 'left-to-right'} LSTM tagger: accuracy on intensifiers {acc_adv:.1%}, on adjectives {acc_adj:.1%}")

Exactly as predicted: both taggers get the adjectives right, but only the bidirectional one can tag *terribly* correctly. The left-to-right tagger is stuck at chance on intensifiers, no matter how long it trains, because the information simply isn't available to it yet. **Use bidirectionality when you have the whole input.**

### 4.4 Multi-layer RNNs
* RNNs are already "deep" on one dimension (they unroll over many timesteps). We can also make them "deep" in another dimension by **applying multiple RNNs**: a **multi-layer** (or **stacked**) RNN.
* This allows the network to compute **more complex representations**. The lower RNNs should compute lower-level features and the higher RNNs higher-level features.
* **The hidden states from RNN layer $i$ are the inputs to RNN layer $i+1$.**

**In practice:**
* High-performing RNNs are usually multi-layer (but aren't as deep as convolutional or feed-forward networks).
* Britz et al. (2017) find that for NMT, **2 to 4 layers** is best for the encoder RNN, and **4 layers** is best for the decoder RNN. Often 2 layers is a lot better than 1, and 3 might be a little better than 2.
* Usually, skip/dense connections are needed to train deeper RNNs (e.g., 8 layers).
* Transformer-based networks (e.g., BERT) are usually deeper, like 12 or 24 layers. They have a lot of skipping-like connections.

With our layer API, stacking is just feeding `Hs` from one layer as `X` to the next (and in the backward pass, feeding layer 2's `dX` as layer 1's `dHs`). The PyTorch version in §7 uses `num_layers=2`.

### 4.5 LSTMs: real-world success
* In 2013–2015, LSTMs started achieving state-of-the-art results: handwriting recognition, speech recognition, machine translation, parsing, image captioning, as well as language models. LSTMs became the dominant approach for most NLP tasks.
* Now (2019–2024), **Transformers have become dominant** for all tasks. E.g., in WMT (a machine translation conference + competition): in WMT 2014 there were **0** neural MT systems (!); the WMT 2016 report contains "RNN" 44 times (and these systems won); the WMT 2019 report has "RNN" 7 times and "Transformer" 105 times.

---
## 5. Machine translation

**Machine translation (MT)** is the task of translating a sentence $x$ from one language (the **source language**) to a sentence $y$ in another language (the **target language**).

> x: *L'homme est né libre, et partout il est dans les fers*  
> y: *Man is born free, but everywhere he is in chains* (Rousseau)

### 5.1 The early history of MT: 1950s
* MT research began in the early 1950s on machines less powerful than high school calculators (before the term "A.I." was coined!), concurrent with foundational work on automata, formal languages, probabilities, and information theory.
* It was heavily funded by the military, but the systems were basically just **simple rule-based systems doing word substitution**.
* Human language is more complicated than that, and varies more across languages! There was little understanding of natural language syntax, semantics, pragmatics. The problem soon appeared intractable. (There's a 1-minute video of a 1954 demo linked from the slides.)

### 5.2 Our toy translation task
To experience the difficulty, we use a toy language pair. The source is English-like (SVO). The target language "Toy" is **verb-final (SOV)**, puts **adjectives after the noun** (in mirrored order), and uses **postpositions**. Words map one-to-one, but **word order changes a lot**, including over long distances: the verb moves past the entire object and every prepositional phrase.

In [ ]:
LEX = {"the": "le", "a": "un", "dog": "kaz", "cat": "miro", "bird": "tupi", "farmer": "olan", "child": "bebi",
       "teacher": "sanu", "ball": "pilo", "book": "libu", "apple": "mazo", "letter": "kato",
       "big": "gros", "small": "pik", "red": "roj", "old": "vej", "happy": "felo",
       "sees": "vidas", "eats": "manjas", "finds": "trovas", "throws": "jetas", "likes": "amas",
       "in": "ene", "near": "apud", "with": "kun", "garden": "jardo", "house": "domo", "park": "parko", ".": "."}
N_ANIM = ["dog", "cat", "bird", "farmer", "child", "teacher"]
N_OBJ = ["ball", "book", "apple", "letter"] + N_ANIM
N_PLACE = ["garden", "house", "park"]
ADJS = ["big", "small", "red", "old", "happy"]
VERBS_MT = ["sees", "eats", "finds", "throws", "likes"]
PREPS = ["in", "near", "with"]

def noun_phrase(nouns, max_adj, rng):
    det = rng.choice(["the", "a"]); adjs = [rng.choice(ADJS) for _ in range(rng.choice(range(max_adj + 1)))]
    n = rng.choice(nouns)
    return [det] + adjs + [n], [LEX[det], LEX[n]] + [LEX[a] for a in reversed(adjs)]   # adjectives follow the noun

def translation_pair(rng=random, max_adj=2, max_pp=2):
    s_subj, t_subj = noun_phrase(N_ANIM, max_adj, rng)
    verb = rng.choice(VERBS_MT)
    s_obj, t_obj = noun_phrase(N_OBJ, max_adj, rng)
    src, tgt = s_subj + [verb] + s_obj, t_subj + t_obj
    for _ in range(rng.choice(range(max_pp + 1))):
        prep = rng.choice(PREPS); s_np, t_np = noun_phrase(N_PLACE + N_ANIM, max_adj, rng)
        src += [prep] + s_np; tgt += t_np + [LEX[prep]]                                  # postpositions
    return src + ["."], tgt + [LEX[verb], "."]                                            # verb-final

random.seed(0)
mt_data = [translation_pair() for _ in range(10000)]
mt_train, mt_test = mt_data[:8500], mt_data[8500:]
for s, t in mt_data[:3]:
    print("EN :", " ".join(s)); print("TOY:", " ".join(t)); print()
print("source length: mean", np.mean([len(s) for s, _ in mt_data]).round(1), " max", max(len(s) for s, _ in mt_data))

**A 1950s-style system: word substitution.** We even give it a *perfect* bilingual dictionary. How often is the translation exactly right?

In [ ]:
def word_substitution(src):
    return [LEX[w] for w in src]

exact = np.mean([word_substitution(s) == t for s, t in mt_test])
same_words = np.mean([sorted(word_substitution(s)) == sorted(t) for s, t in mt_test])
print(f"word-for-word translation, perfect dictionary: exact match {exact:.1%}")
print(f"...even though it has exactly the right bag of words {same_words:.1%} of the time")
print("example:", " ".join(word_substitution(mt_test[0][0])), "\n    gold:", " ".join(mt_test[0][1]))

**Zero.** Every sentence has a verb that must move, so the right words in the wrong order never produce a correct translation. Real language pairs differ in word order, morphology, idioms, and one-to-many word mappings, which is why the problem "soon appeared intractable."

### 5.3 1990s–2010s: statistical machine translation (SMT)
**Core idea:** learn a **probabilistic model from data**. Suppose we're translating French → English. We want to find the best English sentence $y$, given French sentence $x$:
$$\arg\max_y P(y\mid x)$$
Use **Bayes' rule** to break this down into two components to be learned separately:
$$= \arg\max_y \underbrace{P(x\mid y)}_{\text{Translation model}}\ \underbrace{P(y)}_{\text{Language model}}$$
* **Translation model:** models how words and phrases should be translated (**fidelity**). Learned from **parallel data**.
* **Language model:** models how to write good English (**fluency**). Learned from **monolingual data** (an LM from Lecture 5!).

🔎 **How the translation model was learned:** with **alignment**, a latent correspondence between source and target words (IBM Models 1–5, Brown et al. 1993), learned with EM from sentence pairs. Alignments can be one-to-one, one-to-many, many-to-one, many-to-many, or have no counterpart (*spurious* words), and they cross when word order differs (exactly our verb movement). Decoding meant a heuristic search over phrase segmentations, translations, and reorderings.

**What happens in translation isn't trivial to model.** The slides show a Chinese sentence about 600 Spaniards landing in Mexico in 1519 to conquer the Aztec Empire. Google Translate's SMT outputs from 2009, 2013, and 2015 all garble who lost two-thirds of whose soldiers. SMT systems were good at local phrases, but bad at long-distance structure and coherence.

**SMT was a huge research field.** The best systems were **extremely complex**: hundreds of important details, many separately designed subcomponents, lots of feature engineering to capture particular language phenomena, extra resources to compile and maintain (like tables of equivalent phrases), lots of human effort, and **repeated effort for each language pair**!

Then, in **2014**: **Neural Machine Translation** (*dramatic reenactment* on the slides).

---
## 6. What is neural machine translation?

**Neural machine translation (NMT)** is a way to do machine translation with a **single end-to-end neural network**. The neural network architecture is called a **sequence-to-sequence model** (aka **seq2seq**), and it involves **two RNNs**.

### 6.1 The sequence-to-sequence model
* The **encoder RNN** reads the source sentence (*il m' a entarté*) and produces an **encoding** of it. Its final state provides the **initial hidden state for the decoder RNN**.
* The **decoder RNN** is a **language model** that generates the target sentence (*he hit me with a pie*), **conditioned on the encoding**.
* At **test time**, the decoder's output is **fed in as the next step's input**: `<START>` → argmax *he* → feed *he* → argmax *hit* → … → `<END>`.

**Sequence-to-sequence is versatile!** The general notion is an **encoder–decoder** model: one neural network takes input and produces a neural representation; another network produces output based on that representation. If input and output are sequences, it's seq2seq. Many NLP tasks can be phrased this way:
* **Summarization** (long text → short text) · **Dialogue** (previous utterances → next utterance) · **Parsing** (input text → output parse as a sequence) · **Code generation** (natural language → Python code)

### 6.2 NMT is a conditional language model
* **Language model** because the decoder is predicting the next word of the target sentence $y$.
* **Conditional** because its predictions are also conditioned on the source sentence $x$.

NMT directly calculates $P(y\mid x)$:
$$P(y\mid x) = P(y_1\mid x)\,P(y_2\mid y_1, x)\,P(y_3\mid y_1, y_2, x)\cdots P(y_T\mid y_1,\dots,y_{T-1}, x)$$
Each factor is the probability of the next target word, given the target words so far and the source sentence $x$.

**How to train an NMT system?** (Easy) answer: get a **big parallel corpus**. (But there's exciting work on unsupervised NMT, data augmentation, back-translation, etc.)

### 6.3 Training: end to end
Feed the source sentence (from the corpus) to the encoder, and the target sentence, shifted right with `<START>`, to the decoder (**teacher forcing**, as in L5). At each decoder step, $J_t$ = negative log probability of the true next target word. The loss is the average:
$$J = \frac{1}{T}\sum_{t=1}^{T}J_t \qquad (J_1 = -\log P(\text{"he"}),\ \dots,\ J_7 = -\log P(\text{<END>}))$$
**Seq2seq is optimized as a single system. Backpropagation operates "end-to-end":** the gradient flows from the decoder's losses back through the decoder, into the **encoder's final state**, and through the encoder.

Notice the architecture's weak spot: **the entire source sentence must be squeezed into the single vector that conditions the decoder.** The slide labels it **"Conditioning = Bottleneck"**. Keep that in mind as you look at the results.

### 6.4 Implementation from scratch
Encoder LSTM over the (padded) source; decoder LSTM initialized with the encoder's final $(h, c)$; a softmax over the target vocabulary at each step. In the backward pass, the decoder's gradient with respect to its **initial state** becomes the encoder's gradient with respect to its **final state**: that single connection is how the encoder learns.

In [ ]:
src_vocab = ["<pad>"] + sorted({w for s, _ in mt_data for w in s})
tgt_vocab = ["<pad>", "<s>"] + sorted({w for _, t in mt_data for w in t})
S2I = {w: i for i, w in enumerate(src_vocab)}
T2I = {w: i for i, w in enumerate(tgt_vocab)}

def mt_batch(pairs):
    B = len(pairs); Ls = max(len(s) for s, _ in pairs); Lt = max(len(t) for _, t in pairs)
    S = np.zeros((B, Ls), int); SM = np.zeros((B, Ls))
    Tin = np.zeros((B, Lt), int); Tout = np.zeros((B, Lt), int); TM = np.zeros((B, Lt))
    for b, (s, t) in enumerate(pairs):
        S[b, :len(s)] = [S2I[w] for w in s]; SM[b, :len(s)] = 1
        ids = [T2I[w] for w in t]
        Tin[b, :len(t)] = [T2I["<s>"]] + ids[:-1]; Tout[b, :len(t)] = ids; TM[b, :len(t)] = 1
    return S, SM, Tin, Tout, TM

def init_seq2seq(d=32, H=64, seed=0):
    r = np.random.default_rng(seed)
    p = {"Es": r.normal(0, 0.1, (len(src_vocab), d)), "Et": r.normal(0, 0.1, (len(tgt_vocab), d)),
         "U": r.normal(0, 1 / np.sqrt(H), (len(tgt_vocab), H)), "c": np.zeros(len(tgt_vocab))}
    p["encW"], p["encb"] = lstm_init(d, H, r)
    p["decW"], p["decb"] = lstm_init(d, H, r)
    return p

def seq2seq_fb(p, S, SM, Tin, Tout, TM, need_grad=True):
    B = S.shape[0]; H = p["U"].shape[1]; d = p["Es"].shape[1]; z = np.zeros((B, H))
    _, hT, cT, enc_cache = lstm_forward(p["encW"], p["encb"], p["Es"][S], SM, z, z)          # encode
    Hd, _, _, dec_cache = lstm_forward(p["decW"], p["decb"], p["Et"][Tin], np.ones(Tin.shape), hT, cT)  # decode
    P = softmax(Hd @ p["U"].T + p["c"])
    Bi, Ti = np.meshgrid(np.arange(B), np.arange(Tout.shape[1]), indexing="ij")
    ntok = TM.sum()
    loss = -(np.log(P[Bi, Ti, Tout] + 1e-12) * TM).sum() / ntok
    if not need_grad:
        return loss, P
    G = P.copy(); G[Bi, Ti, Tout] -= 1; G *= TM[:, :, None] / ntok
    g = {"U": np.einsum("btv,bth->vh", G, Hd), "c": G.sum((0, 1))}
    dXt, dh0, dc0, g["decW"], g["decb"] = lstm_backward(p["decW"], G @ p["U"], z, z, dec_cache, d)
    # the bottleneck: the encoder only receives gradient through the decoder's initial state
    dXs, _, _, g["encW"], g["encb"] = lstm_backward(p["encW"], np.zeros((B, S.shape[1], H)), dh0, dc0, enc_cache, d)
    g["Et"] = np.zeros_like(p["Et"]); np.add.at(g["Et"], Tin, dXt)
    g["Es"] = np.zeros_like(p["Es"]); np.add.at(g["Es"], S, dXs)
    return loss, g

# gradient check on a tiny model
p_chk = init_seq2seq(d=4, H=3, seed=1)
bt = mt_batch(mt_train[:3])
_, g_chk = seq2seq_fb(p_chk, *bt)
r = np.random.default_rng(0); worst = 0.0
for k in p_chk:
    for _ in range(4):
        idx = tuple(r.integers(0, s) for s in p_chk[k].shape)
        if k == "Es": idx = (bt[0][0, 1], r.integers(0, 4))
        if k == "Et": idx = (bt[2][0, 1], r.integers(0, 4))
        old = p_chk[k][idx]
        p_chk[k][idx] = old + 1e-6; lp = seq2seq_fb(p_chk, *bt, need_grad=False)[0]
        p_chk[k][idx] = old - 1e-6; lm_ = seq2seq_fb(p_chk, *bt, need_grad=False)[0]
        p_chk[k][idx] = old
        worst = max(worst, abs((lp - lm_) / 2e-6 - g_chk[k][idx]))
print(f"seq2seq gradient check: max |analytic - numerical| = {worst:.1e}")

**Greedy decoding** (the test-time behavior in the slide's diagram): run the encoder, then repeatedly take the argmax word and feed it back in, until the end token (here, ".").

In [ ]:
def greedy_decode(p, src, max_len=40):
    H = p["U"].shape[1]
    S = np.array([[S2I[w] for w in src]])
    _, h, c, _ = lstm_forward(p["encW"], p["encb"], p["Es"][S], np.ones(S.shape), np.zeros((1, H)), np.zeros((1, H)))
    y, out = T2I["<s>"], []
    for _ in range(max_len):
        _, h, c, _ = lstm_forward(p["decW"], p["decb"], p["Et"][[[y]]], np.ones((1, 1)), h, c)
        y = int(np.argmax(p["U"] @ h[0] + p["c"]))
        out.append(tgt_vocab[y])
        if tgt_vocab[y] == ".":
            break
    return out

def length_bucket(n):
    return "short (<=10)" if n <= 10 else ("medium (11-15)" if n <= 15 else "long (16+)")

def evaluate_mt(decode_fn, pairs):
    res = collections.defaultdict(list)
    for s, t in pairs:
        res[length_bucket(len(s))].append(decode_fn(s) == t)
    return {k: round(float(np.mean(v)), 3) for k, v in sorted(res.items())}

p_s2s = init_seq2seq()
opt = Adam(p_s2s)
r = np.random.default_rng(0)
t0 = time.time()
for epoch in range(15):
    order = r.permutation(len(mt_train)); total = 0.0
    for k in range(0, len(order), 64):
        loss, g = seq2seq_fb(p_s2s, *mt_batch([mt_train[i] for i in order[k:k + 64]]))
        clip_grads(g); opt.step(p_s2s, g); total += loss
    print(f"epoch {epoch + 1:2d}: train loss {total / math.ceil(len(order) / 64):.4f}   "
          f"exact-match on 300 test sentences: {evaluate_mt(lambda s: greedy_decode(p_s2s, s), mt_test[:300])}"
          f"   ({time.time() - t0:.0f}s)")
np.savez("seq2seq_no_attention.npz", **p_s2s)          # Lecture 7's notebook can reuse this model

In [ ]:
for s, t in mt_test[:4]:
    out = greedy_decode(p_s2s, s)
    print("source   :", " ".join(s))
    print("gold     :", " ".join(t))
    print("seq2seq  :", " ".join(out), "  ✓" if out == t else "  ✗")
    print()

The model has learned the reordering rules from data alone, with no dictionary and no grammar: subject first, adjectives after the noun, postpositions, verb at the end. But look at how accuracy depends on length. **Short sentences work much better than long ones.** The decoder has to recover *every* word, and its position, from one fixed-size vector. That's the **bottleneck**, and Lecture 7's **attention** is the solution: it lets the decoder look directly at every encoder state.

### 6.5 Multi-layer deep encoder–decoder (Sutskever et al. 2014; Luong et al. 2015)
Real NMT systems stacked 2–4 LSTM layers in both the encoder ("builds up sentence meaning") and the decoder (generates the translation, feeding in the last word). The hidden states from RNN layer $i$ are the inputs to layer $i+1$. The conditioning is still the bottleneck. 🔎 Sutskever et al. also found that **reversing the source sentence** helped: it puts the first source words close to the first target words, shortening the gradient paths at the start of the output.

### 6.6 NMT: the first big success story of NLP deep learning
NMT went from a **fringe research attempt in 2014** to the **leading standard method in 2016**:
* 2014: first seq2seq paper published (Sutskever et al. 2014)
* 2016: **Google Translate switches from SMT to NMT**, and by 2018 everyone had (*The Great A.I. Awakening*, NYT Magazine, Dec 2016)
* This was amazing! SMT systems, built by hundreds of engineers over many years, were outperformed by NMT systems trained by small groups of engineers in a few months.

---
## 7. PyTorch versions

### 7.1 LSTM language model with the forget-bias trick
PyTorch packs the gates as **(input, forget, cell, output)**, so the forget-gate bias is the second quarter of `bias_ih_l0` / `bias_hh_l0`. (Both biases are added, so we set one of them.)

In [ ]:
# [PyTorch] LSTM on the long-gap memory task, with and without the forget-bias trick.
import torch
import torch.nn as nn
import torch.nn.functional as F

class MemoryLSTM(nn.Module):
    def __init__(self, forget_bias=None, d=16, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(V_MEM, d)
        self.lstm = nn.LSTM(d, hidden, batch_first=True)
        self.out = nn.Linear(hidden, N_SYM)
        if forget_bias is not None:
            with torch.no_grad():
                H = hidden
                self.lstm.bias_ih_l0[H:2 * H].fill_(forget_bias)     # PyTorch gate order: i, f, g, o
                self.lstm.bias_hh_l0[H:2 * H].fill_(0.0)
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        return self.out(h[-1])

for fb in [None, 3.0]:
    torch.manual_seed(0); model = MemoryLSTM(fb); opt_t = torch.optim.Adam(model.parameters(), lr=3e-3)
    r = np.random.default_rng(0)
    for step in range(1500):
        X, y = memory_batch(T_GAP, 64, r)
        loss = F.cross_entropy(model(torch.tensor(X)), torch.tensor(y))
        opt_t.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt_t.step()
    X, y = memory_batch(T_GAP, 1000, r)
    with torch.no_grad():
        acc = (model(torch.tensor(X)).argmax(1).numpy() == y).mean()
    print(f"PyTorch LSTM, forget bias {'default (~0)' if fb is None else fb}: accuracy at gap {T_GAP} = {acc:.2f}")

### 7.2 Bidirectional, multi-layer LSTM tagger
`bidirectional=True` returns the concatenated $[\overrightarrow h; \overleftarrow h]$ at every position (size `2 * hidden`); `num_layers=2` stacks two (bidirectional) layers, with `dropout` applied between them.

In [ ]:
# [PyTorch] 2-layer BiLSTM tagger on the polarity task.
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, n_tags, d=16, hidden=24, bidirectional=True, layers=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.lstm = nn.LSTM(d, hidden, num_layers=layers, bidirectional=bidirectional, batch_first=True,
                            dropout=0.1 if layers > 1 else 0.0)
        self.out = nn.Linear(hidden * (2 if bidirectional else 1), n_tags)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x))       # (B, T, 2*hidden)
        return self.out(h)

Xt_, Yt_ = torch.tensor(Xtag), torch.tensor(Ytag)
for bidir in [False, True]:
    torch.manual_seed(0); model = BiLSTMTagger(len(tag_vocab), len(TAGS), bidirectional=bidir)
    opt_t = torch.optim.Adam(model.parameters(), lr=5e-3)
    for epoch in range(8):
        perm = torch.randperm(n_tr)
        for k in range(0, n_tr, 50):
            idx = perm[k:k + 50]
            loss = F.cross_entropy(model(Xt_[idx]).reshape(-1, len(TAGS)), Yt_[idx].reshape(-1))
            opt_t.zero_grad(); loss.backward(); opt_t.step()
    model.eval()
    with torch.no_grad():
        pred = model(Xt_[n_tr:]).argmax(-1)
    print(f"{'BiLSTM' if bidir else 'left-to-right LSTM'} (2 layers): intensifier accuracy {(pred[:, 3] == Yt_[n_tr:, 3]).float().mean():.1%}")

### 7.3 Sequence-to-sequence NMT in PyTorch
Same architecture as §6.4. Padding uses index 0 and `ignore_index=0` in the loss. `pack_padded_sequence` lets the encoder stop at each sentence's true length, the equivalent of our mask, so its final state is the state after the last real token.

In [ ]:
# [PyTorch] LSTM encoder-decoder for the toy translation task.
from torch.nn.utils.rnn import pack_padded_sequence

class Seq2Seq(nn.Module):
    def __init__(self, n_src, n_tgt, d=32, hidden=64):
        super().__init__()
        self.src_emb = nn.Embedding(n_src, d, padding_idx=0)
        self.tgt_emb = nn.Embedding(n_tgt, d, padding_idx=0)
        self.encoder = nn.LSTM(d, hidden, batch_first=True)
        self.decoder = nn.LSTM(d, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_tgt)
    def encode(self, src, src_lens):
        packed = pack_padded_sequence(self.src_emb(src), src_lens, batch_first=True, enforce_sorted=False)
        _, state = self.encoder(packed)            # final (h, c) of each sentence = the bottleneck vector
        return state
    def forward(self, src, src_lens, tgt_in):
        state = self.encode(src, src_lens)
        h, _ = self.decoder(self.tgt_emb(tgt_in), state)
        return self.out(h)
    @torch.no_grad()
    def greedy(self, src_ids, max_len=40):
        state = self.encode(torch.tensor([src_ids]), torch.tensor([len(src_ids)]))
        y, out = torch.tensor([[T2I["<s>"]]]), []
        for _ in range(max_len):
            h, state = self.decoder(self.tgt_emb(y), state)
            y = self.out(h[:, -1]).argmax(-1, keepdim=True)
            out.append(tgt_vocab[y.item()])
            if out[-1] == ".":
                break
        return out

torch.manual_seed(0)
model = Seq2Seq(len(src_vocab), len(tgt_vocab))
opt_t = torch.optim.Adam(model.parameters(), lr=2e-3)
for epoch in range(10):
    perm = np.random.default_rng(epoch).permutation(len(mt_train))
    for k in range(0, len(perm), 64):
        S, SM, Tin, Tout, TM = mt_batch([mt_train[i] for i in perm[k:k + 64]])
        logits = model(torch.tensor(S), torch.tensor(SM.sum(1).astype(int)), torch.tensor(Tin))
        loss = F.cross_entropy(logits.reshape(-1, len(tgt_vocab)), torch.tensor(Tout).reshape(-1), ignore_index=0)
        opt_t.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt_t.step()
    model.eval()
    acc = evaluate_mt(lambda s: model.greedy([S2I[w] for w in s]), mt_test[:300])
    model.train()
    print(f"epoch {epoch + 1}: loss {loss.item():.4f}  exact match {acc}")

---
## 8. Summary

Lots of new information today! The practical takeaways:

| Takeaway | What we showed |
|---|---|
| **1. LSTMs are powerful** | An additive cell with gates; the cell path multiplies gradients only by $f$; with forget bias ≈ 1–3 it learns gaps a vanilla RNN can't |
| **2. Clip your gradients** | Used in every training loop here (and the cliff demo in L5) |
| **3. Use bidirectionality when possible** | A left-to-right tagger is stuck at chance on *terribly*; a BiLSTM solves it |
| **4. Encoder–decoder NMT works very well** | Learns reordering from data alone, where perfect word substitution scores 0%, but the single conditioning vector is a **bottleneck** for long sentences |

Also: residual / dense / highway connections help **any** deep network; the GRU is a lighter alternative to the LSTM; stacked RNNs with 2–4 layers are typical; NMT replaced SMT between 2014 and 2016.

### Exercises
1. Derive $\partial c^{(t)}/\partial c^{(t-1)}$ for the LSTM *including* the indirect paths through the gates (which depend on $h^{(t-1)}$, which depends on $c^{(t-1)}$). Why is the direct term $\text{diag}(f^{(t)})$ the important one?
2. Implement a **GRU** layer with the same `forward/backward` interface and gradient-check it. Run it on the memory task.
3. Re-run §2.6 with `T_GAP = 100` and with 5 seeds per model. At what gap does each model start failing? (Expect high variance across seeds for the vanilla RNN.)
4. Make the tagger **2 layers deep** in NumPy by feeding the first BiLSTM's outputs into a second BiLSTM.
5. Implement **source reversal** (Sutskever et al. 2014) in the seq2seq model. Does it help on long sentences?
6. Replace the decoder's initial state with **the mean of all encoder states** instead of the final state. Better or worse?

### References
* Hochreiter & Schmidhuber 1997, *Long Short-Term Memory*; Gers, Schmidhuber & Cummins 2000, *Learning to Forget*.
* Cho et al. 2014 (GRU, RNN encoder–decoder); Chung et al. 2014; Jozefowicz, Zaremba & Sutskever 2015, *An Empirical Exploration of Recurrent Network Architectures*.
* Sutskever, Vinyals & Le 2014, *Sequence to Sequence Learning with Neural Networks*.
* Britz et al. 2017, *Massive Exploration of Neural Machine Translation Architectures*.
* He et al. 2016 (ResNet); Huang et al. 2017 (DenseNet); Srivastava et al. 2015 (Highway Networks).
* Brown et al. 1993, *The Mathematics of Statistical Machine Translation* (IBM models).
* Olah 2015, *Understanding LSTM Networks* (blog).
* Koehn, *Statistical Machine Translation* (book), for the SMT era.

**Next:** Lecture 7, MT evaluation (BLEU), **attention**, and final-project advice.